# expdpy — Analyze panel data

_Notebook version: built 2026-06-22 04:19 UTC — re-open this notebook from GitHub if yours is older, to get the latest version._

A cloud-runnable walkthrough of the **Analyze** module of [expdpy](https://github.com/cmg777/expdpy) on the bundled `kuznets` panel. Run the install cell below first, then run the rest top to bottom.

> The first cell installs everything and then **restarts the Colab runtime once** so the upgraded NumPy loads cleanly. When it reconnects, run the cells again (Runtime > Run all) — the install cell skips the restart the second time.

This notebook mirrors the [Analyze page](https://cmg777.github.io/expdpy/analyze.html) of the docs.

In [ ]:
import importlib.util
import os

!pip install -q "numpy>=2.1" "numba>=0.61" "expdpy @ git+https://github.com/cmg777/expdpy.git"
!pip install -q --force-reinstall --no-deps "expdpy @ git+https://github.com/cmg777/expdpy.git"

_RESTART_FLAG = "/tmp/.expdpy_runtime_restarted"
_ON_COLAB = importlib.util.find_spec("google.colab") is not None
if _ON_COLAB and not os.path.exists(_RESTART_FLAG):
    with open(_RESTART_FLAG, "w"):
        pass
    print("Install complete - restarting the runtime once so NumPy loads cleanly.")
    print("After it reconnects, run the cells again (Runtime > Run all).")
    os.kill(os.getpid(), 9)

In [ ]:
# Ensure Plotly figures render in Google Colab (a no-op in other notebook frontends).
import plotly.io as pio

try:
    import google.colab  # noqa: F401  (present only on Colab)

    pio.renderers.default = "colab"
except ImportError:
    pass

The **Analyze** module estimates models on a panel: OLS with **multi-way fixed effects** and
**clustered standard errors**, **random effects**, **correlated random effects** (the Mundlak
device), the **Frisch–Waugh–Lovell** plot, classic **pooled / between** models, the **Hausman
test**, post-estimation tools, **robust inference**, and **event-study / difference-in-differences**
designs.

In [ ]:
import expdpy as ex
from expdpy.data import load_kuznets, load_kuznets_data_def

# `set_labels` attaches the data dictionary's human-readable labels, so the FWL axes,
# panel view and tables below read with full labels.
df = ex.set_labels(load_kuznets(), load_kuznets_data_def())

## Guided tour

### Regression with fixed effects and clustered SEs

`kuznets` is a country–year panel, so the natural specification absorbs **two-way (country +
year) fixed effects**. A cubic in (log) GDP per capita recovers the N — a positive,
significant cubic term — *within* country, with standard errors clustered by country:

In [ ]:
res = ex.analyze_regression_table(
    df,
    dvs="gini_regional",
    idvs=["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"],
    feffects=["country", "year"],
    clusters=["country"],
)
res.etable

A coefficient plot compares specifications side by side — pooled OLS against two-way fixed
effects:

In [ ]:
pooled = ex.analyze_regression_table(
    df, dvs="gini_regional", idvs=["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"]
)
ex.analyze_coefficient_plot(
    [pooled, res], model_labels=["Pooled OLS", "Two-way FE"]
).fig

Every result can explain itself in plain, **associational** language (see
[Learn](learn.qmd) for the full teaching layer):

In [ ]:
print(res.interpret())

### Frisch–Waugh–Lovell plot

The same coefficient, *seen*. `analyze_fwl_plot` residualizes both the outcome and the focal
regressor on the **other** regressors **and** the fixed effects, then scatters the two
residuals. By the Frisch–Waugh–Lovell theorem the fitted slope equals the focal coefficient
in the table above:

In [ ]:
ex.analyze_fwl_plot(
    df,
    dv="gini_regional",
    var="log_gdp_pc",
    controls=["log_gdp_pc_sq", "log_gdp_pc_cu"],
    feffects=["country", "year"],
    clusters=["country"],
).fig

### Post-estimation

**Predictions** — fitted values, residuals and actuals on the estimation sample:

In [ ]:
ex.analyze_predictions(res).df.head()

**Fixed-effect estimates** — the estimated country intercepts the panel absorbs (top 20):

In [ ]:
ex.analyze_fixef_plot(res, fixef="country", top_n=20).fig

**Joint (Wald) test** — are the curvature terms jointly different from zero?

In [ ]:
#| warning: false
print(
    ex.analyze_joint_test(res, hypotheses=["log_gdp_pc_sq", "log_gdp_pc_cu"]).summary()
)

### Stepwise comparison and panel-friendly standard errors

Beyond the OLS table, `analyze_estimation` adds **stepwise / multiple-outcome** comparison,
serial-correlation-robust standard errors (**Newey–West**, **Driscoll–Kraay**) and weights.
Here, a cumulative-stepwise comparison — watch the estimate move as terms are added:

In [ ]:
ex.analyze_estimation(
    df,
    dv="gini_regional",
    idvs=["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"],
    stepwise="csw",
).etable

### Classic panel models, CRE and the Hausman test

`analyze_panel_table` estimates **pooled / between / fixed / random effects** side by side:

In [ ]:
ex.analyze_panel_table(
    df, dv="gini_regional", idvs=["log_gdp_pc"], entity="country", time="year"
).etable

`analyze_cre_table` adds the **correlated-random-effects (Mundlak)** estimator — a
random-effects model augmented with each regressor's unit mean. The coefficient on the
regressor equals its fixed-effects (within) estimate, while a joint test on the mean terms is
the regression-form Hausman test:

In [ ]:
cre = ex.analyze_cre_table(
    df, dv="gini_regional", idvs=["log_gdp_pc"], entity="country", time="year"
)
print(cre.interpret())

`analyze_hausman_test` chooses between fixed and random effects directly:

In [ ]:
print(
    ex.analyze_hausman_test(
        df, dv="gini_regional", idvs=["log_gdp_pc"], entity="country", time="year"
    ).interpret()
)

### Treatment structure & event study

For treated panels, the bundled `staggered_did` dataset is a ready-made example.
`analyze_panel_view` shows the **treatment structure** (who is treated, when):

In [ ]:
from expdpy.data import load_staggered_did

did = load_staggered_did()
ex.analyze_panel_view(did, unit="unit", time="year", cohort="cohort").fig

`analyze_event_study` traces the **dynamic treatment effect** with a built-in pre-trend
check, using a modern staggered-adoption estimator (Gardner's `did2s` here; `twfe`,
Sun–Abraham `saturated` and `lpdid` are also available):

In [ ]:
ex.analyze_event_study(
    did, outcome="outcome", unit="unit", time="year", cohort="cohort", estimator="did2s"
).fig

### Robust inference

When clusters are few, large-sample cluster-robust p-values can be over-confident.
`analyze_robust_inference` offers **randomization inference** (`ritest`) and the **wild
cluster bootstrap** (`wildboot`, needs the optional `wildboottest` package):

In [ ]:
#| warning: false
did_f = did.assign(treated=did["treated"].astype(float))
m = ex.analyze_regression_table(
    did_f, dvs="outcome", idvs=["treated"], clusters=["unit"]
)
ri = ex.analyze_robust_inference(
    m, param="treated", method="ritest", reps=500, cluster="unit", seed=0
)
print(
    f"Randomization-inference for the treatment effect "
    f"(estimate {ri.estimate:.3f}): p = {ri.p_value:.4f} over {ri.reps} permutations."
)

## Examples

A **basic** and **advanced** example for every Analyze function.

### `analyze_regression_table`

Basic — a pooled OLS regression of the cubic Kuznets curve:

In [ ]:
ex.analyze_regression_table(
    df, dvs="gini_regional", idvs=["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"]
).etable

Advanced — two-way fixed effects with standard errors clustered by country:

In [ ]:
ex.analyze_regression_table(
    df,
    dvs="gini_regional",
    idvs=["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"],
    feffects=["country", "year"],
    clusters=["country"],
).etable

### `analyze_estimation`

OLS with stepwise / multiple-outcome comparison and a rich choice of standard errors
(HC1–HC3, CRV1/CRV3, Newey–West, Driscoll–Kraay) plus weights.

Basic — a cumulative-stepwise comparison:

In [ ]:
ex.analyze_estimation(
    df,
    dv="gini_regional",
    idvs=["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"],
    stepwise="csw",
).etable

Advanced — Driscoll–Kraay standard errors (robust to cross-sectional and serial correlation):

In [ ]:
ex.analyze_estimation(
    df,
    dv="gini_regional",
    idvs=["log_gdp_pc", "log_gdp_pc_sq"],
    vcov="DK",
    time_id="year",
    panel_id="country",
).etable

### `analyze_fwl_plot`

Basic — the partial relationship of the outcome and a single regressor:

In [ ]:
ex.analyze_fwl_plot(df, dv="gini_regional", var="log_gdp_pc").fig

Advanced — residualize on the other cubic terms and two-way fixed effects, with the reported
standard error clustered by country:

In [ ]:
ex.analyze_fwl_plot(
    df,
    dv="gini_regional",
    var="log_gdp_pc",
    controls=["log_gdp_pc_sq", "log_gdp_pc_cu"],
    feffects=["country", "year"],
    clusters=["country"],
).fig

### `analyze_coefficient_plot`

Basic:

In [ ]:
reg = ex.analyze_regression_table(
    df, dvs="gini_regional", idvs=["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"]
)
ex.analyze_coefficient_plot(reg).fig

Advanced — compare pooled OLS and two-way fixed effects, keeping one term:

In [ ]:
fe = ex.analyze_regression_table(
    df, dvs="gini_regional", idvs=["log_gdp_pc"], feffects=["country", "year"]
)
poolg = ex.analyze_regression_table(df, dvs="gini_regional", idvs=["log_gdp_pc"])
ex.analyze_coefficient_plot(
    [poolg.models[0], fe.models[0]],
    model_labels=["Pooled OLS", "Two-way FE"],
    keep=["log_gdp_pc"],
).fig

### `analyze_fixef_plot`

The estimated fixed effects of a model, ranked:

In [ ]:
country_fe = ex.analyze_regression_table(
    df, dvs="gini_regional", idvs=["log_gdp_pc"], feffects=["country", "year"]
)
ex.analyze_fixef_plot(country_fe, fixef="country", top_n=20).fig

### `analyze_predictions`

In [ ]:
ex.analyze_predictions(reg).df.head()

### `analyze_joint_test`

In [ ]:
joint = ex.analyze_joint_test(reg, ["log_gdp_pc", "log_gdp_pc_sq", "log_gdp_pc_cu"])
print(joint.summary())

### `analyze_panel_table`

Pooled / between / fixed / random effects side by side:

In [ ]:
ex.analyze_panel_table(
    df, dv="gini_regional", idvs=["log_gdp_pc"], entity="country", time="year"
).etable

### `analyze_cre_table`

The correlated-random-effects (Mundlak) estimator — its `log_gdp_pc` coefficient equals the
fixed-effects (within) estimate:

In [ ]:
ex.analyze_cre_table(
    df, dv="gini_regional", idvs=["log_gdp_pc"], entity="country", time="year"
).df

### `analyze_hausman_test`

In [ ]:
print(
    ex.analyze_hausman_test(
        df, dv="gini_regional", idvs=["log_gdp_pc"], entity="country", time="year"
    ).interpret()
)

### `analyze_robust_inference`

In [ ]:
clustered = ex.analyze_regression_table(
    df,
    dvs="gini_regional",
    idvs=["log_gdp_pc"],
    feffects=["country"],
    clusters=["country"],
)
ri = ex.analyze_robust_inference(
    clustered, "log_gdp_pc", method="ritest", reps=200, seed=0
)
ri.p_value, ri.conf_int

### `analyze_panel_view`

In [ ]:
ex.analyze_panel_view(did, unit="unit", time="year", cohort="cohort").fig

### `analyze_event_study`

Basic — Gardner's `did2s`:

In [ ]:
es = ex.analyze_event_study(
    did, outcome="outcome", unit="unit", time="year", cohort="cohort", estimator="did2s"
)
print(es.interpret())
es.fig

Advanced — the Sun–Abraham (`saturated`) estimator, one curve per treatment cohort:

In [ ]:
ex.analyze_event_study(
    did,
    outcome="outcome",
    unit="unit",
    time="year",
    cohort="cohort",
    estimator="saturated",
).fig

## Where to go next

- [Learn panel data](learn.qmd) — the ideas behind fixed effects, demeaning, first
  differences and CRE, with runnable sandboxes.
- [Explore panel data](explore.qmd) — the EDA that precedes a model.
- Prefer no code? Launch the [Analyze app](streamlit.qmd).